# Interactive Agent Session: Chapter 1 — AutoGen Multi-Agent Boardroom 🏢

**The Mission:** Coordinate 5 specialized agents to audit global stock markets in a virtual boardroom.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet pyautogen yfinance python-dotenv

In [ ]:
%%capture captured_output

import os
import autogen
from dotenv import load_dotenv

load_dotenv()

llm_config = {
    "config_list": [{"model": "gpt-4o-mini", "api_key": os.getenv("OPENAI_API_KEY")}],
    "temperature": 0
}

# 1. Human Proxy & Execution Environment
user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    system_message="Human administrator. Execute the python code written by the engineers and return the output to the chat.",
    code_execution_config={"work_dir": "coding", "use_docker": False},
    human_input_mode="NEVER"
)

# 2. Company Employees (Agents)
ceo = autogen.AssistantAgent(
    name="CEO",
    system_message="You are the CEO of an AI Financial Firm. Your goal is to orchestrate the team to produce a flawless financial report. You assign tasks to the Data_Engineer, Financial_Analyst, QA_Tester, and Risk_Manager. You synthesize their work into a final executive summary.",
    llm_config=llm_config,
)

data_eng = autogen.AssistantAgent(
    name="Data_Engineer",
    system_message="You are a Data Engineer. Write executable Python code (using the yfinance library) to fetch the requested stock data and print it. IMPORTANT: Use Close column, NOT Adj Close as newer yfinance versions removed it. Never write bash or pip commands, only pure Python.",
    llm_config=llm_config,
)

analyst = autogen.AssistantAgent(
    name="Financial_Analyst",
    system_message="You are a Senior Financial Analyst. Read the raw terminal data provided by the Data Engineer, calculate volatility or trends, and write a detailed market analysis.",
    llm_config=llm_config,
)

qa_tester = autogen.AssistantAgent(
    name="QA_Tester",
    system_message="You are the QA Tester. Review the Financial Analyst calculations. If there are math errors, missing data points, or logical fallacies, explicitly tell them to rewrite it. Do not accept the report until it is mathematically sound.",
    llm_config=llm_config,
)

risk_manager = autogen.AssistantAgent(
    name="Risk_Manager",
    system_message="You are the Chief Risk Officer. Once QA approves the analysis, review the report and highlight downside risks, geopolitical impacts, and explicitly assign a final Risk Score from 1-10.",
    llm_config=llm_config,
)

# 3. Virtual Office (Group Chat)
groupchat = autogen.GroupChat(
    agents=[user_proxy, ceo, data_eng, analyst, qa_tester, risk_manager],
    messages=[],
    max_round=15
)
manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

# 4. Start the business day (output is captured by %%capture)
user_proxy.initiate_chat(
    manager,
    message="We need a comprehensive Q3 investment report for MSFT and TSLA. Please fetch the data, analyze it, rigorously test the findings for math errors, and assign a risk score."
)

print("Done! Run the next cell to see the Premium Boardroom UI.")

In [ ]:
from IPython.display import display, HTML

def render_premium_boardroom(history):
    colors = {"CEO": "#6366f1", "Data_Engineer": "#22c55e", "Financial_Analyst": "#eab308", "QA_Tester": "#ef4444", "Risk_Manager": "#a855f7", "User_Proxy": "#64748b"}
    msgs_html = ""
    for m in history:
        name = m.get("name", "System")
        content = m.get("content", "") or ""
        color = colors.get(name, "#64748b")
        content_safe = content.replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        msgs_html += '<div style="margin-bottom:20px; border-left:4px solid ' + color + '; padding-left:20px; background:#111827; padding:20px; border-radius:0 15px 15px 0;">'
        msgs_html += '<div style="display:flex; align-items:center; gap:8px; margin-bottom:10px;">'
        msgs_html += '<div style="width:8px; height:8px; border-radius:50%; background:' + color + ';"></div>'
        msgs_html += '<b style="color:' + color + '; text-transform:uppercase; font-size:10px; letter-spacing:2px;">' + name + '</b>'
        msgs_html += '</div>'
        msgs_html += '<div style="font-size:13px; color:#e2e8f0; line-height:1.6;">' + content_safe[:2000] + '</div>'
        msgs_html += '</div>'
    html = '<style>@import url("https://fonts.googleapis.com/css2?family=Inter:wght@400;700&family=Unbounded:wght@800&display=swap");</style>'
    html += '<div style="max-width:900px; margin:30px auto; font-family:Inter,sans-serif; background:#000; padding:40px; border-radius:40px; border:1px solid #1a1e2e; color:#fff; box-shadow:0 50px 100px rgba(0,0,0,0.8);">'
    html += '<div style="border-bottom:1px solid #1a1e2e; padding-bottom:25px; margin-bottom:30px; display:flex; align-items:center; justify-content:space-between;">'
    html += '<div style="font-family:Unbounded,sans-serif; font-size:18px; color:#6366f1; letter-spacing:1px;">BOARDROOM_SYNC: VIRTUAL_FIRM</div>'
    html += '<div style="font-size:11px; background:#6366f122; color:#6366f1; padding:4px 12px; border-radius:50px;">SESSION_COMPLETE</div>'
    html += '</div>'
    html += msgs_html
    html += '</div>'
    display(HTML(html))

render_premium_boardroom(groupchat.messages)